← [Overview](00_overview.ipynb)

# Extreme periods

**Why extremes matter:** capacity sizing in energy-system models requires the
**peak demand** and **minimum generation** periods to survive the aggregation.
A centroid averaging over all sunny days will dampen the solar peak; a clustering
of 365 days into 6 clusters might never pick the single highest-load day.
tsam's `ExtremeConfig` forces specific extreme periods into the representative set.

## The three strategies

| Method | Effect on cluster count |
|---|---|
| `append` | +1 per extreme period (total count increases) |
| `replace` | 0 (replaces the nearest cluster center) |
| `new_cluster` | +1 per extreme, affected periods reassigned |

## Selection criteria

* `max_value` / `min_value` — period containing the single maximum or minimum timestep
* `max_period` / `min_period` — period with the highest or lowest sum

**TSAM configuration for extreme periods:**

In [ ]:
import tsam
from tsam import ClusterConfig, ExtremeConfig

# ExtremeConfig is passed as the extremes= argument to tsam.aggregate.
# method controls how the extreme period is incorporated:
#   'append'      — add as extra cluster (total count increases by 1 per extreme)
#   'replace'     — replace nearest existing cluster center (count stays same)
#   'new_cluster' — add as new cluster and reassign affected periods (count +1)

# Selection criteria (at least one must be non-empty):
#   max_value  — period containing the single highest timestep value
#   min_value  — period containing the single lowest timestep value
#   max_period — period with the highest column sum (e.g. peak solar day)
#   min_period — period with the lowest column sum (e.g. lowest wind day)

cfg_append = ExtremeConfig(
    method="append",
    max_value=["load"],  # preserve the period with the peak load timestep
)
cfg_replace = ExtremeConfig(method="replace", max_value=["load"])
cfg_new_cluster = ExtremeConfig(method="new_cluster", max_value=["load"])

# Example with multiple criteria:
cfg_multi = ExtremeConfig(
    method="append",
    max_value=["load"],
    min_period=["solar"],
)

print("append config:     ", cfg_append)
print("replace config:    ", cfg_replace)
print("new_cluster config:", cfg_new_cluster)
print("multi config:      ", cfg_multi)

# Full aggregate call:
# result = tsam.aggregate(
#     df,
#     n_clusters=k,
#     period_duration="1D",
#     cluster=ClusterConfig(method="hierarchical"),
#     extremes=ExtremeConfig(method="append", max_value=["load"]),
# )

In [ ]:
import pandas as pd
import plotly.io as pio

from tsam import ExtremeConfig

pio.renderers.default = "notebook_connected"

# --------------------------------------------------------------------------
# Load the shared tiny dataset produced by 01_preprocessing (../tiny.csv).
# --------------------------------------------------------------------------
tiny = pd.read_csv("../tiny.csv", index_col=0, parse_dates=True)

# Real dataset
raw = pd.read_csv("../testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]
UNITS = {"GHI": "W/m²", "T": "°C", "Wind": "m/s", "Load": "MW"}
print("tiny:", tiny.shape, "  real:", data.shape)

---

## The problem: peaks are averaged away

In [ ]:
# day_5 has the extreme load peak (value=10). Does it survive without ExtremeConfig?
print("Max load by day in tiny dataset:")
print(tiny["load"].groupby(tiny.index.date).max().rename("max_load").to_string())
print("\nGlobal max:", tiny["load"].max(), "(day_5, timestep t3)")

In [ ]:
# Without extremes: peak day may be averaged away
result_noext = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="kmeans"),
)
rep_max_noext = result_noext.cluster_representatives["load"].max()
print("Without extremes — max load in representatives:", rep_max_noext)
print("  (original max was 10; centroid averaging smooths it out)")

---

## Strategy 1: `append`

Adds the extreme period as an **extra cluster**. The total number of representative
periods increases by one for each extreme specified.

In [ ]:
result_append = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="kmeans"),
    extremes=ExtremeConfig(method="append", max_value=["load"]),
)
print("append: n_clusters =", result_append.n_clusters, "(3 regular + 1 extreme)")
print(
    "Max load in representatives:", result_append.cluster_representatives["load"].max()
)
print("Cluster counts:", result_append.cluster_counts)

---

## Strategy 2: `replace`

**Replaces** the nearest existing cluster center with the extreme period. The total
cluster count stays the same.

In [ ]:
result_replace = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="kmeans"),
    extremes=ExtremeConfig(method="replace", max_value=["load"]),
)
print(
    "replace: n_clusters =",
    result_replace.n_clusters,
    "(unchanged — extreme replaced nearest center)",
)
print(
    "Max load in representatives:", result_replace.cluster_representatives["load"].max()
)
print("Cluster counts:", result_replace.cluster_counts)

---

## Strategy 3: `new_cluster`

Adds the extreme as a new cluster **and** reassigns all periods that were formerly
in the replaced cluster. Cluster count increases by 1.

In [ ]:
result_newcluster = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="kmeans"),
    extremes=ExtremeConfig(method="new_cluster", max_value=["load"]),
)
print("new_cluster: n_clusters =", result_newcluster.n_clusters)
print(
    "Max load in representatives:",
    result_newcluster.cluster_representatives["load"].max(),
)
print("Cluster counts:", result_newcluster.cluster_counts)

---

## Real-data example: preserve peak-Load and minimum-Wind days

In [ ]:
result_ext_real = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
    extremes=ExtremeConfig(
        method="append",
        max_value=["Load"],
        min_period=["Wind"],
    ),
)
print("Extreme result — n_clusters:", result_ext_real.n_clusters)
print("Cluster counts:", result_ext_real.cluster_counts)

result_ext_real.plot.compare(
    columns=["Load"],
    mode="duration_curve",
    title="Duration curve: hierarchical + extreme periods vs original",
)

### The extreme days on the calendar

With `append`, each requested extreme (here the peak-Load day and the minimum-Wind day) becomes its own one-member cluster — pick them out as the lone colours among the regular clusters.

In [ ]:
result_ext_real.plot.clusters_over_time(
    columns=["Load"], units=UNITS, title="Hierarchical + appended extreme days"
)

In [ ]:
# Summary: compare strategies on real data
rows = []
for method in ["append", "replace", "new_cluster"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical"),
        extremes=ExtremeConfig(method=method, max_value=["Load"]),
    )
    rows.append(
        {
            "extreme_method": method,
            "n_clusters": r.n_clusters,
            "weighted_rmse": round(r.accuracy.weighted_rmse, 4),
            "max_Load_in_reps": round(r.cluster_representatives["Load"].max(), 2),
        }
    )

# Also add no-extremes baseline
r_base = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
)
rows.insert(
    0,
    {
        "extreme_method": "none",
        "n_clusters": r_base.n_clusters,
        "weighted_rmse": round(r_base.accuracy.weighted_rmse, 4),
        "max_Load_in_reps": round(r_base.cluster_representatives["Load"].max(), 2),
    },
)

print("Extreme strategies comparison (hierarchical k=6, real dataset):")
pd.DataFrame(rows)

---

**Up next:**
* [Rescaling & denormalisation](05_rescaling.ipynb) — restore the column totals (extreme clusters are left untouched) and return to physical units

**See also:**
* [Extreme periods how-to](../how-to/extreme_periods.ipynb) — detailed guide with all selection criteria
* [Representation](03_representation.ipynb) — how representatives are formed before extremes are applied